<a href="https://colab.research.google.com/github/Ambaright/ST-554-Homework-6/blob/main/Baright_HW6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: Amanda Baright

Date: 03.11.2026

Purpose: ST 554 Homework 6

# Part I: More Practice Querying a Database

For the SQL querying practice, we will again be using the Major League Baseball database. Information on downloading the database can be found [here](https://github.com/jknecht/baseball-archive-sqlite).

## Q1: Connecting to Database

The first thing we need to do is connect to the database and look at all of the tables in the database. We'll use the `read_sql()` function from `pandas` to return the result as a dataframe. We will also need to import `sqlite3` and `pandas`, and upload the `sqlite` file into the JupyterNotebook environment.

In [11]:
import sqlite3
import pandas as pd
con = sqlite3.connect("lahman_1871-2022.sqlite")
pd.read_sql("SELECT * FROM sqlite_schema WHERE type = 'table';", con)

,type,name,tbl_name,rootpage,sql
0,table,AllstarFull,AllstarFull,2,"CREATE TABLE AllstarFull (\nplayerID TEXT,\nye..."
1,table,Appearances,Appearances,3,"CREATE TABLE Appearances (\nyearID INTEGER,\nt..."
2,table,AwardsManagers,AwardsManagers,4,"CREATE TABLE AwardsManagers (\nplayerID TEXT,\..."
3,table,AwardsPlayers,AwardsPlayers,5,"CREATE TABLE AwardsPlayers (\nplayerID TEXT,\n..."
4,table,AwardsShareManagers,AwardsShareManagers,6,CREATE TABLE AwardsShareManagers (\nawardID TE...
5,table,AwardsSharePlayers,AwardsSharePlayers,7,CREATE TABLE AwardsSharePlayers (\nawardID TEX...
6,table,Batting,Batting,8,"CREATE TABLE Batting (\nplayerID TEXT,\nyearID..."
7,table,BattingPost,BattingPost,9,"CREATE TABLE BattingPost (\nyearID INTEGER,\nr..."
8,table,CollegePlaying,CollegePlaying,10,"CREATE TABLE CollegePlaying (\nplayerID TEXT,\..."
9,table,Fielding,Fielding,11,"CREATE TABLE Fielding (\nplayerID TEXT,\nyearI..."


## Q2: Hall of Fame Pitchers

Now using SQL, we will construct a table of hall of fame pitchers that gives the `playerID` and their total (sum) for `GS`, `G`, `W`, `L`, `IPOuts`, `CG`, `SHO`, and `SV` columns. One thing we want to keep in mind is that we want stats on both the regular and post season games for pitchers that were inducted into the Hall of Fame, this requires aggregating these two tables with a `UNION`. My approach was to use the `UNION` on two separate table queries, with each query having a `WHERE` condition that looks to see if the `playerID` can be found in the `HallOfFame` table and where they were inducted. We will then use `pandas` to group by the `playerID` and do the summing.

In [12]:
pitcher = pd.read_sql("""
SELECT playerID, GS, G, W, L, IPOuts, CG, SHO, SV
FROM Pitching
WHERE playerID IN (SELECT playerID FROM HallOfFame WHERE inducted = 'Y')

UNION

SELECT playerID, GS, G, W, L, IPOuts, CG, SHO, SV
FROM PitchingPost
WHERE playerID IN (SELECT playerID FROM HallOfFame WHERE inducted = 'Y');
""", con)

# using pandas we can get the total summary stats
pitcher.groupby('playerID').sum()


,GS,G,W,L,IPouts,CG,SHO,SV
playerID,,,,,,,,
alexape01,604,703,376,210,15699,441,90,33
ansonca01,0,3,0,1,12,0,0,1
becklja01,1,1,0,1,12,0,0,0
bendech01,344,469,218,131,9306,264,41,34
blylebe01,691,700,292,251,15052,243,60,0
...,...,...,...,...,...,...,...,...
willivi01,472,515,249,206,12023,388,50,11
wrighge01,0,3,0,1,15,0,0,0
wrighha01,8,36,4,4,301,0,0,14


## Q3: Pitchers Batting Stats

Now for all of the Hall of Famer pitchers, we want to use SQL to create a table of their batting statistics. That is, include their `playerID` and their total (sum) for `AB`, `R`, `H`, `HR`, `RBI`, `BB`, and `SO`. Again, we will want the regular and post seasons statistics, so we can use a `UNION` to join these two statistics. Then we will do the summing again using `pandas` and `groupby()`.

In [14]:
pitcherBatting = pd.read_sql("""
SELECT playerID, AB, R, H, HR, RBI, BB, SO
FROM Batting
WHERE playerID IN (SELECT playerID FROM HallOfFame WHERE inducted = 'Y')

UNION

SELECT playerID, AB, R, H, HR, RBI, BB, SO
FROM BattingPost
WHERE playerID IN (SELECT playerID FROM HallOfFame WHERE inducted = 'Y');
""",con)

# again using pandas for the total summary stats
pitcherBatting.groupby('playerID').sum()

,AB,R,H,HR,RBI,BB,SO
playerID,,,,,,,
aaronha01,12433,2185,3796,761,2313.0,1407,1396.0
alexape01,1823,155,379,11,164.0,77,279.0
alomaro01,9303,1540,2796,214,1167.0,1059,1172.0
alstowa01,1,0,0,0,0.0,0,1.0
andersp01,477,42,104,0,34.0,42,53.0
...,...,...,...,...,...,...,...
wynnea01,1711,136,367,17,174.0,141,333.0
yastrca01,12053,1831,3443,456,1855.0,1854,1396.0
youngcy01,2986,327,625,18,293.0,81,389.0
